In [30]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from tavily import TavilyClient

from langgraph.checkpoint.memory import InMemorySaver

from dotenv import load_dotenv

load_dotenv()

True

In [31]:
model = init_chat_model(model="gpt-5-nano")

In [32]:
@tool
def web_search(query: str):

    """
    Search for the query on the internet
    """

    tavily_client = TavilyClient()

    search = tavily_client.search(query)

    return search

In [33]:
system_prompt = """
You are an expert chef who helps people find recipes on the internet.

If there's an image, analyze the image then give response accrdingly.

Use your tools to search the internet for the recipes. 

Use the web_search tool then give response. Give precise response. Short responses.
"""

chef_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver())

In [34]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.webp', multiple=False)
display(uploader)

FileUpload(value=(), accept='.webp', description='Upload')

In [35]:
print(uploader.value)

({'name': 'fridge-5452069.webp', 'type': 'image/webp', 'size': 47388, 'content': <memory at 0x000001A091571D80>, 'last_modified': datetime.datetime(2026, 8, 2, 19, 7, 31, 221000, tzinfo=datetime.timezone.utc)},)


In [36]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [37]:
question = HumanMessage(content=[
    {"type":"text", "text":"these are some leftovers in the fridge. what should I make?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

config = {"configurable":{"thread_id":2}}

result = chef_agent.invoke(
    {
        "messages":[question]
    },
    config=config,
)

In [38]:
result

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'these are some leftovers in the fridge. what should I make?'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='350b8f56-ea22-491c-957c-484c006950a4'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 958, 'prompt_tokens': 1029, 'total_tokens': 1987, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 832, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E8VwEiLeldmTHgc0WmpGTR4YIT0s6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fc3e6-aac4-7d90-9c0d-92092efc8d11-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'shakshuka with peppers